# Config


In [ ]:
%run <EDU_Config>


In [ ]:
# Deployment parameters
# Reporting period for attendance opportunities. period_end is exclusive.
period_start = "{PERIOD_START}"
period_end = "{PERIOD_END}"

# An unset reporting period cannot be defaulted: the period decides which enrolled
# instructional days become attendance opportunities, so guessing it would silently
# change every attendance denominator. Fail here with a readable message rather than
# letting an empty string surface later as an opaque DATE cast error.
for _name, _value in (("period_start", period_start), ("period_end", period_end)):
    if not _value or _value.startswith("{"):
        raise ValueError(
            f"{_name} is not set. Install with -PeriodStart and -PeriodEnd, or set both "
            "in this parameter cell, using YYYY-MM-DD. period_end is exclusive."
        )
if period_start >= period_end:
    raise ValueError(
        f"period_end ({period_end}) must be later than period_start ({period_start}); the end is exclusive."
    )

# One UTC cutoff shared by every table in this refresh.
as_of = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
logging.info(f"Silver: {silver_lakehouse_name} → Gold: {gold_lakehouse_name}")
logging.info(f"Reporting period [{period_start}, {period_end}) | as_of {as_of}")


# Validate Silver

Gold reads Silver and never writes to it. The run stops before the first Gold write if an
input contract is broken.


In [ ]:
required_silver_tables = [
    "School", "Student", "Teacher", "Course", "Section", "LearningContent",
    "SchoolEnrollment", "SectionEnrollment", "TeacherAssignment", "CalendarDay",
    "Attendance", "AssessmentInstrument", "AssessmentWindow", "AssessmentResult",
    "Obligation", "SubmissionState", "Submission", "ActionType", "InterventionAction",
    "AcademicTerm", "SchoolAddress",
]

missing_tables = [
    table for table in required_silver_tables
    if not table_exists(get_silver_full_table_name(table))
]
if missing_tables:
    raise GoldDataQualityError(f"Silver is not ready. Missing tables: {missing_tables}")

display(get_table_counts({t: get_silver_full_table_name(t) for t in required_silver_tables}))


In [ ]:
# Pin every input to its current Delta version before the first read, so the run sees one
# consistent set of versions and the snapshot fingerprint describes exactly what it read.
silver_versions = pin_silver_snapshot(required_silver_tables)
snapshot_id = get_snapshot_id(TRANSFORMATION_VERSION, period_start, period_end, silver_versions)
logging.info(f"Transformation {TRANSFORMATION_VERSION} | snapshot {snapshot_id[:12]}")
display(spark.createDataFrame(sorted(silver_versions.items()), ["SilverTable", "DeltaVersion"]))


In [ ]:
silver = get_silver_view_name

# An interrupted earlier run can leave a candidate behind. Clear it before staging a new
# one, so the tables hold the published snapshot plus at most this run's candidate.
discard_unpublished_snapshots(GOLD_TARGETS)
logging.info(f"Published snapshot before this run: {get_published_snapshot_id()}")

# Keys and relationships
assert_unique_key(silver("School"), ["school_id"])
assert_unique_key(silver("Student"), ["student_id"])
assert_unique_key(silver("Section"), ["section_id"])
assert_unique_key(silver("LearningContent"), ["content_id"])
assert_unique_key(silver("Obligation"), ["obligation_id"])
assert_unique_key(silver("SubmissionState"), ["state_id"])
assert_unique_key(silver("Submission"), ["submission_id"])
# The newest attempt per obligation has to be a single row, because it decides which
# attempt counts towards achievement. Asserting the pair here makes a tie fail loudly on
# the input rather than being resolved arbitrarily in the transformation.
assert_unique_key(silver("Submission"), ["obligation_id", "attempt"])
# The course title is the reporting subject, so it must resolve to one row per course.
assert_unique_key(silver("Course"), ["course_id"])
assert_unique_key(silver("AssessmentResult"), ["result_id"])
assert_unique_key(silver("CalendarDay"), ["school_id", "instructional_date"])
assert_unique_key(silver("Attendance"), ["student_id", "school_id", "attendance_date"])
assert_unique_key(silver("ActionType"), ["action_type_id"])
assert_unique_key(silver("AcademicTerm"), ["term_id"])
assert_unique_key(silver("SchoolAddress"), ["school_id"])
assert_unique_key(silver("InterventionAction"), ["action_id"])

assert_foreign_key(silver("Section"), "school_id", silver("School"), "school_id")
assert_foreign_key(silver("Section"), "course_id", silver("Course"), "course_id")
assert_foreign_key(silver("LearningContent"), "section_id", silver("Section"), "section_id")
assert_foreign_key(silver("Obligation"), "content_id", silver("LearningContent"), "content_id")
assert_foreign_key(silver("SubmissionState"), "obligation_id", silver("Obligation"), "obligation_id")
assert_foreign_key(silver("Submission"), "obligation_id", silver("Obligation"), "obligation_id")
assert_foreign_key(silver("AssessmentResult"), "assessment_id", silver("AssessmentInstrument"), "assessment_id")
assert_foreign_key(silver("AssessmentResult"), "window_id", silver("AssessmentWindow"), "window_id")
assert_foreign_key(silver("InterventionAction"), "action_type_id", silver("ActionType"), "action_type_id")
assert_foreign_key(silver("SchoolAddress"), "school_id", silver("School"), "school_id")
assert_foreign_key(silver("InterventionAction"), "student_id", silver("Student"), "student_id")
assert_foreign_key(silver("InterventionAction"), "school_id", silver("School"), "school_id")
assert_foreign_key(silver("InterventionAction"), "teacher_id", silver("Teacher"), "teacher_id")

# Membership history
assert_valid_intervals(silver("SchoolEnrollment"), ["student_id"], "enrollment_id")
assert_valid_intervals(silver("SectionEnrollment"), ["student_id", "section_id"], "enrollment_id")
assert_valid_intervals(silver("TeacherAssignment"), ["teacher_id", "section_id"], "assignment_id")

# Required values must be present before anything is derived from them. A null key or
# date would otherwise pass the key and foreign-key checks, which both ignore nulls.
assert_not_null(silver("School"), ["school_id", "name"])
assert_not_null(silver("Student"), ["student_id"])
assert_not_null(silver("Teacher"), ["teacher_id"])
assert_not_null(silver("Section"), ["section_id", "school_id", "course_id"])
assert_not_null(silver("LearningContent"), ["content_id", "section_id"])
assert_not_null(silver("CalendarDay"), ["school_id", "instructional_date"])
assert_not_null(silver("SchoolEnrollment"), ["enrollment_id", "student_id", "school_id", "grade", "valid_from", "valid_to"])
assert_not_null(silver("SectionEnrollment"), ["enrollment_id", "student_id", "section_id", "valid_from", "valid_to"])
assert_not_null(silver("TeacherAssignment"), ["assignment_id", "teacher_id", "section_id", "valid_from", "valid_to"])
assert_not_null(silver("Attendance"), ["student_id", "school_id", "attendance_date", "status"])
assert_not_null(silver("AssessmentResult"), ["result_id", "student_id", "school_id", "assessment_date", "assessment_id", "window_id"])
assert_not_null(silver("Obligation"), ["obligation_id", "student_id", "content_id", "section_id"])
assert_not_null(silver("SubmissionState"), ["state_id", "obligation_id", "state"])
assert_not_null(silver("Submission"), ["submission_id", "obligation_id", "student_id", "content_id", "attempt", "submitted_at"])
assert_not_null(silver("InterventionAction"), ["action_id", "student_id", "school_id", "teacher_id", "action_type_id", "action_date", "reason", "reason_category", "status"])

# Only mapped attendance codes may be interpreted; an unmapped code is not "unknown",
# and a null status is unmapped rather than absent, so reject it explicitly. NOT IN
# never matches a null, so the null case has to be named separately.
assert_no_rows(
    "Attendance uses mapped statuses",
    f"""
    SELECT * FROM {silver('Attendance')}
    WHERE status IS NULL
       OR status NOT IN ('present', 'absent', 'excused', 'unknown')
    """,
)
assert_no_rows(
    "Attendance falls on an instructional day the learner was enrolled for",
    f"""
    SELECT a.*
    FROM {silver('Attendance')} AS a
    WHERE NOT EXISTS (
        SELECT 1
        FROM {silver('CalendarDay')} AS c
        JOIN {silver('SchoolEnrollment')} AS m
          ON c.school_id = m.school_id
         AND c.instructional_date >= m.valid_from
         AND c.instructional_date < m.valid_to
        WHERE c.school_id = a.school_id
          AND m.student_id = a.student_id
          AND c.instructional_date = a.attendance_date
    )
    """,
)
assert_no_rows(
    "Assessment administrations fall inside school membership",
    f"""
    SELECT r.*
    FROM {silver('AssessmentResult')} AS r
    WHERE NOT EXISTS (
        SELECT 1 FROM {silver('SchoolEnrollment')} AS m
        WHERE m.student_id = r.student_id
          AND m.school_id = r.school_id
          AND r.assessment_date >= m.valid_from
          AND r.assessment_date < m.valid_to
    )
    """,
)
# The submission's own learner and task must be the ones the obligation records.
# Achievement is assembled from both sides of this pair — the mark scheme comes from
# the task, the section and subject from the obligation — so a disagreeing row would
# publish a hybrid fact that every key and row-count check still accepts. Null-safe
# equality, because <> is null when either side is null and a null comparison is not a
# match, which would let the very rows with a missing key through unexamined.
assert_no_rows(
    "Submissions agree with their obligation",
    f"""
    SELECT s.*
    FROM {silver('Submission')} AS s
    JOIN {silver('Obligation')} AS o ON s.obligation_id = o.obligation_id
    WHERE NOT (s.student_id <=> o.student_id)
       OR NOT (s.content_id <=> o.content_id)
    """,
)

# A teacher action is only reportable for a learner the school and, when given, the class
# actually held on that day, logged by a teacher of that class. Anything else would put an
# action on a page that the learner or teacher does not belong to.
assert_no_rows(
    "Teacher actions use mapped statuses",
    f"""
    SELECT * FROM {silver('InterventionAction')}
    WHERE status NOT IN ('open', 'completed')
       OR (status = 'completed' AND completed_date IS NULL)
    """,
)
assert_no_rows(
    "Teacher actions fall inside school membership",
    f"""
    SELECT a.*
    FROM {silver('InterventionAction')} AS a
    WHERE NOT EXISTS (
        SELECT 1 FROM {silver('SchoolEnrollment')} AS m
        WHERE m.student_id = a.student_id
          AND m.school_id = a.school_id
          AND a.action_date >= m.valid_from
          AND a.action_date < m.valid_to
    )
    """,
)
assert_no_rows(
    "Class-level teacher actions belong to the learner's class and its teacher",
    f"""
    SELECT a.*
    FROM {silver('InterventionAction')} AS a
    WHERE a.section_id IS NOT NULL
      AND (
        NOT EXISTS (
            SELECT 1 FROM {silver('SectionEnrollment')} AS e
            WHERE e.student_id = a.student_id
              AND e.section_id = a.section_id
              AND a.action_date >= e.valid_from
              AND a.action_date < e.valid_to
        )
        OR NOT EXISTS (
            SELECT 1 FROM {silver('TeacherAssignment')} AS t
            WHERE t.teacher_id = a.teacher_id
              AND t.section_id = a.section_id
              AND a.action_date >= t.valid_from
              AND a.action_date < t.valid_to
        )
      )
    """,
)

# School years must not overlap, or a day would belong to two years and a year-on-year
# comparison would count it twice.
assert_no_rows(
    "School years do not overlap",
    f"""
    SELECT a.term_id
    FROM {silver('AcademicTerm')} AS a
    JOIN {silver('AcademicTerm')} AS b
      ON a.term_id < b.term_id
     AND a.valid_from < b.valid_to
     AND b.valid_from < a.valid_to
    """,
)

logging.info("✅ Silver input contracts validated.")


# Enrichment


## Dimensions


### Build: DimSchool


In [ ]:
def EnrichDimSchool() -> DataFrame:
    # The postal address comes from SchoolAddress; a school without one keeps its row and
    # falls back to the city recorded on School.
    school = get_silver_table("School")
    address = get_silver_table("SchoolAddress").select(
        "school_id", "street_address", col("city").alias("address_city"), "state", "postal_code", "county"
    )
    return (
        school.join(address, on="school_id", how="left")
        .select(
            "school_id", "district_id", "name", "street_address",
            coalesce(col("address_city"), col("city")).alias("city"),
            "state", "postal_code", "county",
        )
    )

write_gold_table(EnrichDimSchool(), "DimSchool", snapshot_id, as_of)


### Build: DimStudent


In [ ]:
def EnrichDimStudent() -> DataFrame:
    # The display name is for authorized teacher and school views only; it is never an
    # identity-matching key and never leaves the school scope.
    return (
        get_silver_table("Student")
        .select(
            col("student_id"),
            concat_ws(" ", col("given_name"), col("surname")).alias("display_name"),
        )
    )

write_gold_table(EnrichDimStudent(), "DimStudent", snapshot_id, as_of)


### Build: DimTeacher


In [ ]:
def EnrichDimTeacher() -> DataFrame:
    return (
        get_silver_table("Teacher")
        .select("teacher_id", "name")
    )

write_gold_table(EnrichDimTeacher(), "DimTeacher", snapshot_id, as_of)


### Build: DimSection

The course title is the reporting subject. There is no separate curriculum or standards
source, so no skill or topic is inferred from it.


In [ ]:
def EnrichDimSection() -> DataFrame:
    section = get_silver_table("Section")
    course = get_silver_table("Course").select(
        col("course_id"), col("name").alias("subject")
    )

    return (
        section.join(course, on="course_id", how="inner")
        .select(
            col("section_id"),
            col("school_id"),
            col("course_id"),
            col("subject"),
            col("grade"),
            col("name"),
        )
    )

write_gold_table(EnrichDimSection(), "DimSection", snapshot_id, as_of)


### Build: DimContent

`due_at` is the base assignment due time. The source has no learner-specific override, so
a null stays undated rather than becoming overdue.


In [ ]:
def EnrichDimContent() -> DataFrame:
    return (
        get_silver_table("LearningContent")
        .select(
            "content_id",
            "section_id",
            "name",
            "source_name",
            "description",
            "due_at",
            "max_score",
        )
    )

write_gold_table(EnrichDimContent(), "DimContent", snapshot_id, as_of)


### Build: DimAssessment


In [ ]:
def EnrichDimAssessment() -> DataFrame:
    # unit records the score scale (RIT). Scales are never mixed or rescaled.
    return (
        get_silver_table("AssessmentInstrument")
        .select("assessment_id", "subject", "unit", "name")
    )

write_gold_table(EnrichDimAssessment(), "DimAssessment", snapshot_id, as_of)


### Build: DimWindow


In [ ]:
def EnrichDimWindow() -> DataFrame:
    return (
        get_silver_table("AssessmentWindow")
        .select("window_id", "name")
    )

write_gold_table(EnrichDimWindow(), "DimWindow", snapshot_id, as_of)


### Build: DimDate

The calendar spans the dates actually present in the source: instructional days, base
assignment due dates, assessment dates and teacher action dates. A populated date range is not evidence that
attendance was captured for those days.


In [ ]:
def EnrichDimDate() -> DataFrame:
    observed_dates = (
        get_silver_table("CalendarDay").select(col("instructional_date").alias("date"))
        .unionByName(
            get_silver_table("LearningContent")
            .filter(col("due_at").isNotNull())
            .select(to_date(col("due_at")).alias("date"))
        )
        .unionByName(
            get_silver_table("AssessmentResult").select(col("assessment_date").alias("date"))
        )
        .unionByName(
            get_silver_table("InterventionAction").select(col("action_date").alias("date"))
        )
    )

    bounds = observed_dates.agg(
        min("date").alias("first_date"), max("date").alias("last_date")
    ).first()

    if bounds["first_date"] is None:
        raise GoldDataQualityError("Cannot build DimDate: no dated Silver records.")

    return (
        spark.sql(
            f"SELECT explode(sequence(DATE '{bounds['first_date']}', DATE '{bounds['last_date']}', INTERVAL 1 DAY)) AS date"
        )
    )

write_gold_table(EnrichDimDate(), "DimDate", snapshot_id, as_of)


## Bridges

Class membership is kept as half-open `[valid_from, valid_to)` intervals so that a
teacher or class filter resolves the learners who were enrolled on each business day.
Facts are filtered through these bridges; bridge rows are never joined into a fact.


### Build: BridgeStudentSection


In [ ]:
def EnrichBridgeStudentSection() -> DataFrame:
    return (
        get_silver_table("SectionEnrollment")
        .select("enrollment_id", "student_id", "section_id", "valid_from", "valid_to")
    )

write_gold_table(EnrichBridgeStudentSection(), "BridgeStudentSection", snapshot_id, as_of)


### Build: BridgeTeacherSection


In [ ]:
def EnrichBridgeTeacherSection() -> DataFrame:
    return (
        get_silver_table("TeacherAssignment")
        .select("assignment_id", "teacher_id", "section_id", "valid_from", "valid_to")
    )

write_gold_table(EnrichBridgeTeacherSection(), "BridgeTeacherSection", snapshot_id, as_of)


## Facts


### Build: FactAttendanceDay

Every instructional day a learner was enrolled for becomes one expected opportunity, then
the observation is joined in. A missing row is unknown, not absence, and an excused
absence is not presence. The join is on school membership only, so a learner with six
classes still contributes a single row per day.

`rolling_absent_days` counts the learner's absences in their last
`ATTENDANCE_RULE_WINDOW_DAYS` instructional days in this school and school year, ending with
the row's day, and `rolling_window_complete` says the window is full. The window never reaches
back across the summer into the previous school year. An early-warning rule such as "5 or
more absences in 20 school days" is then a filter on one row. How many absences meet the
rule is chosen in the report, not here, so Gold flags nobody.


In [ ]:
def EnrichFactAttendanceDay() -> DataFrame:
    calendar_day = silver("CalendarDay")
    school_enrollment = silver("SchoolEnrollment")
    attendance = silver("Attendance")
    academic_term = silver("AcademicTerm")

    return spark.sql(f"""
        WITH daily AS (
        SELECT
            sha2(to_json(array('AttendanceDay', m.student_id, c.school_id, CAST(c.instructional_date AS STRING))), 256) AS attendance_day_id,
            c.school_id,
            m.student_id,
            c.instructional_date AS date,
            m.grade,
            CAST(1 AS BIGINT) AS expected_days,
            CAST(CASE WHEN a.status IN ('{ATTENDANCE_PRESENT}', '{ATTENDANCE_ABSENT}', '{ATTENDANCE_EXCUSED}') THEN 1 ELSE 0 END AS BIGINT) AS known_days,
            CAST(CASE WHEN a.status = '{ATTENDANCE_PRESENT}' THEN 1 ELSE 0 END AS BIGINT) AS present_days,
            CAST(CASE WHEN a.status IN ('{ATTENDANCE_ABSENT}', '{ATTENDANCE_EXCUSED}') THEN 1 ELSE 0 END AS BIGINT) AS absent_days,
            CAST(CASE WHEN a.status IN ('{ATTENDANCE_PRESENT}', '{ATTENDANCE_ABSENT}', '{ATTENDANCE_EXCUSED}') THEN 0 ELSE 1 END AS BIGINT) AS unknown_days,
            t.term_id AS term_key
        FROM {calendar_day} AS c
        JOIN {school_enrollment} AS m
          ON c.school_id = m.school_id
         AND c.instructional_date >= m.valid_from
         AND c.instructional_date < m.valid_to
        LEFT JOIN {attendance} AS a
          ON a.school_id = c.school_id
         AND a.student_id = m.student_id
         AND a.attendance_date = c.instructional_date
        LEFT JOIN {academic_term} AS t
          ON c.instructional_date >= t.valid_from
         AND c.instructional_date < t.valid_to
        WHERE c.instructional_date >= DATE '{period_start}'
          AND c.instructional_date < DATE '{period_end}'
        ),
        rolling AS (
        SELECT
            *,
            SUM(expected_days) OVER learner_window AS rolling_expected_days,
            SUM(absent_days) OVER learner_window AS rolling_absent_days
        FROM daily
        WINDOW learner_window AS (
            PARTITION BY student_id, school_id, term_key
            ORDER BY date
            ROWS BETWEEN {ATTENDANCE_RULE_WINDOW_DAYS - 1} PRECEDING AND CURRENT ROW
        )
        )
        SELECT
            attendance_day_id, school_id, student_id, date, grade,
            expected_days, known_days, present_days, absent_days, unknown_days,
            rolling_expected_days, rolling_absent_days,
            (rolling_expected_days = {ATTENDANCE_RULE_WINDOW_DAYS}) AS rolling_window_complete
        FROM rolling
    """)

write_gold_table(EnrichFactAttendanceDay(), "FactAttendanceDay", snapshot_id, as_of)


### Build: FactAssessment

One row per recorded administration. The grade comes from the membership that contains
the test date, not from the current roster. Counting rows counts administrations, not
learners, and a null score stays null instead of becoming a failed result. `is_scored`
states whether a score is present, so no downstream measure has to decide what a null
means.


In [ ]:
def EnrichFactAssessment() -> DataFrame:
    assessment_result = silver("AssessmentResult")
    school_enrollment = silver("SchoolEnrollment")

    return spark.sql(f"""
        SELECT
            r.result_id,
            r.school_id,
            r.student_id,
            r.assessment_date AS date,
            m.grade,
            r.assessment_id,
            r.window_id,
            r.score,
            r.standard_error,
            (r.score IS NOT NULL) AS is_scored
        FROM {assessment_result} AS r
        JOIN {school_enrollment} AS m
          ON m.student_id = r.student_id
         AND m.school_id = r.school_id
         AND r.assessment_date >= m.valid_from
         AND r.assessment_date < m.valid_to
    """)

write_gold_table(EnrichFactAssessment(), "FactAssessment", snapshot_id, as_of)


### Build: FactSubmissionState

The recorded state of each learner task, as provided by the source snapshot. Native
Canvas flags are carried through unchanged; Moodle supplies no equivalent flags, so those
stay null instead of being classified. The observation time of the snapshot is not
available, so this is not a history of weekly states.


In [ ]:
def EnrichFactSubmissionState() -> DataFrame:
    state = get_silver_table("SubmissionState")
    obligation = get_silver_table("Obligation").select(
        "obligation_id", "student_id", "content_id", "section_id"
    )

    return (
        state.join(obligation, on="obligation_id", how="inner")
        .select(
            "state_id",
            "obligation_id",
            "student_id",
            "content_id",
            "section_id",
            "state",
            "source_missing",
            "source_late",
        )
    )

write_gold_table(EnrichFactSubmissionState(), "FactSubmissionState", snapshot_id, as_of)


### Build: FactSubmission

Attempts are kept separately from states so that a learner with several attempts is not
counted several times in the state list. A zero score is a real grade; an unavailable one
stays null. The learner, the task and the class come from the obligation, as they do for
states, and the mark scheme is looked up by that obligation's task, so one validated row
decides every derived column instead of a mixture of two.

Five columns carry coursework achievement so that "how is this learner doing on graded
work" can be answered from this table alone:

- `points_possible` is the mark scheme from the assignment, and only when it is positive.
  A zero or absent maximum is not a denominator, so it stays null rather than producing a
  division by zero or a 0/0 that would read as failure.
- `is_graded` says a score and a usable maximum are both present. It is the filter that
  keeps unmarked work out of an achievement ratio without turning it into a zero.
- `is_late` is the source system's own flag, carried through unchanged. Nothing is
  derived from the due date, because per-learner extensions are not in the data.
- `is_latest_attempt` marks the newest attempt for each obligation, so a resubmission is
  not counted twice. The Silver contract asserts one row per obligation and attempt, so a
  tie fails the run instead of being resolved arbitrarily here.
- `subject` is the section's course title, resolved once here so per-subject achievement
  does not need a two-hop join in every query.

They are operands, not rates. Points earned and points possible are summed separately and
divided in the semantic model, so an achievement figure is a ratio of sums rather than an
average of per-task percentages.


In [ ]:
def EnrichFactSubmission() -> DataFrame:
    submission = silver("Submission")
    obligation = silver("Obligation")
    learning_content = silver("LearningContent")
    section = silver("Section")
    course = silver("Course")

    # Every join is to an asserted unique key, so none of them can multiply a submission;
    # the row count against Silver below proves it.
    #
    # The learner, the task and the section all come from the obligation, and the mark
    # scheme is looked up by the obligation's content, so every derived column hangs off
    # the same validated row. Reaching through the submission's own copies instead would
    # let a Silver row that disagrees with its obligation publish a fact whose mark scheme
    # belongs to one task and whose subject belongs to another. The input contract asserts
    # that the two agree, so this changes nothing on valid input and removes the shape of
    # the failure on invalid input, which FactSubmissionState already does.
    #
    # COALESCE(max_score, 0) > 0 rather than max_score > 0: in SQL a comparison with null
    # is null, and "true AND null" is null, which would leave is_graded null on exactly
    # the rows with no mark scheme. Those rows are not graded, so they must read false.
    return spark.sql(f"""
        SELECT
            s.submission_id,
            s.obligation_id,
            o.student_id,
            o.content_id,
            o.section_id,
            s.attempt,
            s.submitted_at,
            s.score,
            s.state,
            CASE WHEN c.max_score > 0 THEN c.max_score END AS points_possible,
            (s.score IS NOT NULL AND COALESCE(c.max_score, 0) > 0) AS is_graded,
            s.source_late AS is_late,
            (s.attempt = MAX(s.attempt) OVER (PARTITION BY s.obligation_id)) AS is_latest_attempt,
            k.name AS subject
        FROM {submission} AS s
        JOIN {obligation} AS o
          ON o.obligation_id = s.obligation_id
        JOIN {learning_content} AS c
          ON c.content_id = o.content_id
        JOIN {section} AS sec
          ON sec.section_id = o.section_id
        JOIN {course} AS k
          ON k.course_id = sec.course_id
    """)

write_gold_table(EnrichFactSubmission(), "FactSubmission", snapshot_id, as_of)


### Build: FactStudentAction

One row per action a teacher logged for a learner in the reporting period: a check-in, a
family contact, an attendance plan, a counselor referral or a missing-work follow-up. The
learner's year group comes from the membership that contains the action date, and the class
name and subject are resolved once here when the action belongs to a class.

Each action carries the learner's attendance in the `FOLLOW_UP_WINDOW_DAYS` instructional
days before it and the same number after it, as expected, known and absent day components.
The day of the action is in neither window. `is_comparable` says both windows are full, so a
before/after figure only compares equal windows. `is_first_in_category` marks the learner's
first action for the same kind of reason in the period, so a learner with several actions is
counted once in a follow-up figure.

For an action that belongs to a class, the work columns hold the `FOLLOW_UP_WORK_PIECES`
pieces of work in that class due before the action and the same number due after it, and how
many of each have no recorded hand-in. Pieces rather than days, because a class can set
nothing for two weeks. `is_work_comparable` says both sides are full. The attendance windows
and the first-action flag stay inside one school year.

These columns describe what was observed after an action. They are not evidence that the
action caused a change: a rule that fires on a run of absences selects a window that is
unusually high, so part of any fall afterwards is regression to the mean.

In [ ]:
def EnrichFactStudentAction() -> DataFrame:
    action = silver("InterventionAction")
    action_type = silver("ActionType")
    school_enrollment = silver("SchoolEnrollment")
    section = silver("Section")
    course = silver("Course")
    # The staged candidate, never the physical table, for the same reason as the aggregate
    # below: during a refresh the physical table also holds the published snapshot.
    attendance_day = get_candidate_view_name("FactAttendanceDay")
    academic_term = silver("AcademicTerm")
    obligation = silver("Obligation")
    submission = silver("Submission")
    window = FOLLOW_UP_WINDOW_DAYS
    pieces = FOLLOW_UP_WORK_PIECES

    return spark.sql(f"""
        WITH days AS (
            SELECT
                d.student_id, d.school_id, d.date, d.expected_days, d.known_days, d.absent_days,
                t.term_id AS term_key,
                ROW_NUMBER() OVER (PARTITION BY d.student_id, d.school_id, t.term_id ORDER BY d.date) AS day_number
            FROM {attendance_day} AS d
            LEFT JOIN {academic_term} AS t
              ON d.date >= t.valid_from
             AND d.date < t.valid_to
        ),
        actions AS (
            SELECT
                a.action_id, a.student_id, a.school_id, a.section_id, a.teacher_id,
                a.action_date, a.reason, a.reason_category, a.status,
                a.follow_up_date, a.completed_date, a.outcome_note,
                t.name AS action_type,
                m.grade,
                sec.name AS class_name,
                k.name AS subject,
                term.term_id AS term_key
            FROM {action} AS a
            JOIN {action_type} AS t
              ON t.action_type_id = a.action_type_id
            JOIN {school_enrollment} AS m
              ON m.student_id = a.student_id
             AND m.school_id = a.school_id
             AND a.action_date >= m.valid_from
             AND a.action_date < m.valid_to
            LEFT JOIN {section} AS sec
              ON sec.section_id = a.section_id
            LEFT JOIN {course} AS k
              ON k.course_id = sec.course_id
            LEFT JOIN {academic_term} AS term
              ON a.action_date >= term.valid_from
             AND a.action_date < term.valid_to
            WHERE a.action_date >= DATE '{period_start}'
              AND a.action_date < DATE '{period_end}'
        ),
        anchored AS (
            -- The first instructional day on or after the action anchors both windows.
            SELECT a.action_id, a.student_id, a.school_id, a.term_key, MIN(d.day_number) AS action_day
            FROM actions AS a
            JOIN days AS d
              ON d.student_id = a.student_id
             AND d.school_id = a.school_id
             AND d.term_key <=> a.term_key
             AND d.date >= a.action_date
            GROUP BY a.action_id, a.student_id, a.school_id, a.term_key
        ),
        windows AS (
            SELECT
                x.action_id,
                SUM(CASE WHEN d.day_number < x.action_day THEN d.expected_days ELSE 0 END) AS before_expected_days,
                SUM(CASE WHEN d.day_number < x.action_day THEN d.known_days ELSE 0 END) AS before_known_days,
                SUM(CASE WHEN d.day_number < x.action_day THEN d.absent_days ELSE 0 END) AS before_absent_days,
                SUM(CASE WHEN d.day_number > x.action_day THEN d.expected_days ELSE 0 END) AS after_expected_days,
                SUM(CASE WHEN d.day_number > x.action_day THEN d.known_days ELSE 0 END) AS after_known_days,
                SUM(CASE WHEN d.day_number > x.action_day THEN d.absent_days ELSE 0 END) AS after_absent_days
            FROM anchored AS x
            JOIN days AS d
              ON d.student_id = x.student_id
             AND d.school_id = x.school_id
             AND d.term_key <=> x.term_key
             AND d.day_number BETWEEN x.action_day - {window} AND x.action_day + {window}
             AND d.day_number <> x.action_day
            GROUP BY x.action_id
        ),
        work AS (
            -- A piece of work is missing when the learner's obligation has no recorded hand-in.
            SELECT
                o.obligation_id, o.student_id, o.section_id, TO_DATE(o.due_at) AS due_date,
                CASE WHEN s.obligation_id IS NULL THEN 1 ELSE 0 END AS not_handed_in
            FROM {obligation} AS o
            LEFT JOIN (SELECT DISTINCT obligation_id FROM {submission}) AS s
              ON s.obligation_id = o.obligation_id
            WHERE o.due_at IS NOT NULL
        ),
        work_ranked AS (
            SELECT
                a.action_id,
                (w.due_date < a.action_date) AS is_before,
                w.not_handed_in,
                ROW_NUMBER() OVER (
                    PARTITION BY a.action_id, (w.due_date < a.action_date)
                    ORDER BY ABS(DATEDIFF(w.due_date, a.action_date)), w.obligation_id
                ) AS piece
            FROM actions AS a
            JOIN work AS w
              ON w.student_id = a.student_id
             AND w.section_id = a.section_id
             AND w.due_date <> a.action_date
        ),
        work_windows AS (
            SELECT
                action_id,
                SUM(CASE WHEN is_before AND piece <= {pieces} THEN 1 ELSE 0 END) AS before_work_due,
                SUM(CASE WHEN is_before AND piece <= {pieces} THEN not_handed_in ELSE 0 END) AS before_work_missing,
                SUM(CASE WHEN NOT is_before AND piece <= {pieces} THEN 1 ELSE 0 END) AS after_work_due,
                SUM(CASE WHEN NOT is_before AND piece <= {pieces} THEN not_handed_in ELSE 0 END) AS after_work_missing
            FROM work_ranked
            GROUP BY action_id
        )
        SELECT
            a.action_id,
            a.student_id,
            a.school_id,
            a.section_id,
            a.teacher_id,
            a.action_date AS date,
            a.grade,
            a.class_name,
            a.subject,
            a.action_type,
            a.reason,
            a.reason_category,
            a.status,
            a.follow_up_date,
            a.completed_date,
            a.outcome_note,
            (a.status = 'completed') AS is_completed,
            (ROW_NUMBER() OVER (
                PARTITION BY a.student_id, a.reason_category, a.term_key
                ORDER BY a.action_date, a.action_id
            ) = 1) AS is_first_in_category,
            CAST({window} AS BIGINT) AS window_days,
            CAST(COALESCE(w.before_expected_days, 0) AS BIGINT) AS before_expected_days,
            CAST(COALESCE(w.before_known_days, 0) AS BIGINT) AS before_known_days,
            CAST(COALESCE(w.before_absent_days, 0) AS BIGINT) AS before_absent_days,
            CAST(COALESCE(w.after_expected_days, 0) AS BIGINT) AS after_expected_days,
            CAST(COALESCE(w.after_known_days, 0) AS BIGINT) AS after_known_days,
            CAST(COALESCE(w.after_absent_days, 0) AS BIGINT) AS after_absent_days,
            (COALESCE(w.before_expected_days, 0) = {window}
             AND COALESCE(w.after_expected_days, 0) = {window}) AS is_comparable,
            CAST({pieces} AS BIGINT) AS work_window_pieces,
            CAST(COALESCE(ww.before_work_due, 0) AS BIGINT) AS before_work_due,
            CAST(COALESCE(ww.before_work_missing, 0) AS BIGINT) AS before_work_missing,
            CAST(COALESCE(ww.after_work_due, 0) AS BIGINT) AS after_work_due,
            CAST(COALESCE(ww.after_work_missing, 0) AS BIGINT) AS after_work_missing,
            (COALESCE(ww.before_work_due, 0) = {pieces}
             AND COALESCE(ww.after_work_due, 0) = {pieces}) AS is_work_comparable
        FROM actions AS a
        LEFT JOIN windows AS w
          ON w.action_id = a.action_id
        LEFT JOIN work_windows AS ww
          ON ww.action_id = a.action_id
    """)

write_gold_table(EnrichFactStudentAction(), "FactStudentAction", snapshot_id, as_of)

### Build: FactEnrollment

One row per school membership per school year it overlaps, for the school years that touch the
reporting period. `entry_type` compares the learner with the previous school year in the data:
`Continuing` at the same school, `Transferred in` from another school, `New` when they were in
none of the schools, and `First year in data` when there is no earlier year to compare with.
`exit_type` compares with the next school year the same way: `Continuing`, `Transferred out`,
`Completed final grade` when they were in the highest year group their school had that year,
`Left` otherwise, and `Latest year in data` for the most recent year. The school year label is
the calendar year, or both years when the term crosses New Year.

Membership is compared, never predicted: a learner counted as `Left` is not enrolled in any of
the schools in the data the following year, which says nothing about why.

In [ ]:
def EnrichFactEnrollment() -> DataFrame:
    school_enrollment = silver("SchoolEnrollment")
    academic_term = silver("AcademicTerm")

    return spark.sql(f"""
        WITH term AS (
            SELECT
                term_id,
                valid_from AS term_start,
                valid_to AS term_end,
                LAG(term_id) OVER (ORDER BY valid_from) AS prev_term_id,
                LEAD(term_id) OVER (ORDER BY valid_from) AS next_term_id,
                CASE
                    WHEN YEAR(valid_from) = YEAR(DATE_SUB(valid_to, 1)) THEN CAST(YEAR(valid_from) AS STRING)
                    ELSE CONCAT(CAST(YEAR(valid_from) AS STRING), '-', CAST(YEAR(DATE_SUB(valid_to, 1)) AS STRING))
                END AS school_year
            FROM {academic_term}
        ),
        stint AS (
            SELECT
                m.enrollment_id, m.student_id, m.school_id, m.grade, m.valid_from, m.valid_to,
                t.term_id, t.term_start, t.term_end, t.prev_term_id, t.next_term_id, t.school_year,
                TRY_CAST(m.grade AS INT) AS grade_number,
                MAX(TRY_CAST(m.grade AS INT)) OVER (PARTITION BY m.school_id, t.term_id) AS top_grade
            FROM {school_enrollment} AS m
            JOIN term AS t
              ON m.valid_from < t.term_end
             AND m.valid_to > t.term_start
        ),
        presence AS (SELECT DISTINCT student_id, term_id, school_id FROM stint),
        any_presence AS (SELECT DISTINCT student_id, term_id FROM stint)
        SELECT
            sha2(to_json(array('EnrollmentTerm', g.enrollment_id, g.term_id)), 256) AS enrollment_term_id,
            g.enrollment_id, g.student_id, g.school_id, g.grade, g.term_id, g.school_year,
            g.term_start, g.term_end, g.valid_from, g.valid_to,
            CASE
                WHEN g.prev_term_id IS NULL THEN 'First year in data'
                WHEN ps.student_id IS NOT NULL THEN 'Continuing'
                WHEN pa.student_id IS NOT NULL THEN 'Transferred in'
                ELSE 'New'
            END AS entry_type,
            CASE
                WHEN g.next_term_id IS NULL THEN 'Latest year in data'
                WHEN ns.student_id IS NOT NULL THEN 'Continuing'
                WHEN na.student_id IS NOT NULL THEN 'Transferred out'
                WHEN g.grade_number = g.top_grade THEN 'Completed final grade'
                ELSE 'Left'
            END AS exit_type,
            COALESCE(g.grade_number = g.top_grade, FALSE) AS is_final_grade,
            (g.valid_from > g.term_start) AS joined_mid_year,
            (g.valid_to < g.term_end) AS left_mid_year
        FROM stint AS g
        LEFT JOIN presence AS ps
          ON ps.student_id = g.student_id AND ps.term_id = g.prev_term_id AND ps.school_id = g.school_id
        LEFT JOIN any_presence AS pa
          ON pa.student_id = g.student_id AND pa.term_id = g.prev_term_id
        LEFT JOIN presence AS ns
          ON ns.student_id = g.student_id AND ns.term_id = g.next_term_id AND ns.school_id = g.school_id
        LEFT JOIN any_presence AS na
          ON na.student_id = g.student_id AND na.term_id = g.next_term_id
        WHERE g.term_end > DATE '{period_start}'
          AND g.term_start < DATE '{period_end}'
    """)

write_gold_table(EnrichFactEnrollment(), "FactEnrollment", snapshot_id, as_of)

## Aggregate


### Build: AggSchoolAttendance

The district briefing reads this school-grade-period aggregate instead of learner rows.
The day components are summed independently, so an attendance rate is a ratio of sums
rather than an average of school rates. `students` is a distinct count and must not be
summed across rows.


In [ ]:
def EnrichAggSchoolAttendance() -> DataFrame:
    # Read the staged candidate, never the physical table: during a refresh the physical
    # table still holds the published snapshot too, and summing both would double every
    # day component while leaving the distinct student count looking correct.
    attendance_day = get_candidate_view_name("FactAttendanceDay")

    return spark.sql(f"""
        SELECT
            sha2(to_json(array('SchoolAttendance', school_id, grade, '{period_start}', '{period_end}')), 256) AS aggregate_id,
            school_id,
            grade,
            DATE '{period_start}' AS period_start,
            DATE '{period_end}' AS period_end,
            COUNT(DISTINCT student_id) AS students,
            CAST(SUM(present_days) AS BIGINT) AS present_days,
            CAST(SUM(known_days) AS BIGINT) AS known_days,
            CAST(SUM(expected_days) AS BIGINT) AS expected_days,
            CAST(SUM(unknown_days) AS BIGINT) AS unknown_days
        FROM {attendance_day}
        GROUP BY school_id, grade
    """)

write_gold_table(EnrichAggSchoolAttendance(), "AggSchoolAttendance", snapshot_id, as_of)


# Validation

The refresh is only published after these checks pass.


In [ ]:
refreshed_tables = GOLD_TARGETS

gold = get_candidate_view_name
register_candidate_views(refreshed_tables, snapshot_id)

# Validation reads the candidate snapshot only. The previously published snapshot is
# still in the same physical tables at this point and must not be mixed into the checks.
for table_name in refreshed_tables:
    candidate_rows = spark.table(get_candidate_view_name(table_name)).count()
    if candidate_rows == 0:
        raise GoldDataQualityError(
            f"{get_candidate_view_name(table_name)} is empty; the candidate snapshot is incomplete."
        )

# Required Gold values, so a null key cannot slip past the key and foreign-key checks.
assert_not_null(gold("DimSchool"), ["school_id", "name", "snapshot_id", "as_of"])
assert_not_null(gold("DimStudent"), ["student_id", "display_name"])
assert_not_null(gold("DimSection"), ["section_id", "school_id"])
assert_not_null(gold("DimContent"), ["content_id", "section_id"])
assert_not_null(gold("DimDate"), ["date"])
assert_not_null(gold("FactAttendanceDay"), [
    "attendance_day_id", "school_id", "student_id", "date", "grade",
    "expected_days", "known_days", "present_days", "absent_days", "unknown_days",
])
assert_not_null(gold("FactAssessment"), ["result_id", "school_id", "student_id", "date", "assessment_id", "window_id", "is_scored"])
assert_not_null(gold("FactSubmissionState"), ["state_id", "obligation_id", "student_id", "content_id", "state"])
assert_not_null(gold("FactSubmission"), [
    "submission_id", "obligation_id", "student_id", "content_id", "attempt", "submitted_at",
    "is_graded", "is_latest_attempt", "subject",
])
assert_not_null(gold("AggSchoolAttendance"), ["aggregate_id", "school_id", "grade", "period_start", "period_end"])
assert_not_null(gold("FactAttendanceDay"), ["rolling_expected_days", "rolling_absent_days", "rolling_window_complete"])
assert_not_null(gold("FactStudentAction"), [
    "work_window_pieces", "before_work_due", "before_work_missing", "after_work_due",
    "after_work_missing", "is_work_comparable",
])
assert_not_null(gold("FactEnrollment"), [
    "enrollment_term_id", "enrollment_id", "student_id", "school_id", "grade", "term_id",
    "school_year", "term_start", "term_end", "entry_type", "exit_type", "is_final_grade",
])
assert_not_null(gold("FactStudentAction"), [
    "action_id", "student_id", "school_id", "teacher_id", "date", "grade", "action_type",
    "reason", "reason_category", "status", "is_completed", "is_first_in_category",
    "before_expected_days", "after_expected_days", "is_comparable",
])

# Every table belongs to this refresh; a leftover row would mean a partial rewrite.
for table_name in refreshed_tables:
    assert_no_rows(
        f"{gold(table_name)} belongs to snapshot {snapshot_id[:12]}",
        f"SELECT * FROM {gold(table_name)} WHERE snapshot_id <> '{snapshot_id}'",
    )


assert_unique_key(gold("DimSchool"), ["school_id"])
assert_unique_key(gold("DimStudent"), ["student_id"])
assert_unique_key(gold("DimSection"), ["section_id"])
assert_unique_key(gold("DimContent"), ["content_id"])
assert_unique_key(gold("DimDate"), ["date"])
assert_unique_key(gold("FactAttendanceDay"), ["attendance_day_id"])
assert_unique_key(gold("FactAssessment"), ["result_id"])
assert_unique_key(gold("FactSubmissionState"), ["state_id"])
assert_unique_key(gold("FactSubmission"), ["submission_id"])
assert_unique_key(gold("AggSchoolAttendance"), ["aggregate_id"])
assert_unique_key(gold("FactStudentAction"), ["action_id"])
assert_unique_key(gold("FactEnrollment"), ["enrollment_term_id"])

assert_foreign_key(gold("DimSection"), "school_id", gold("DimSchool"), "school_id")
assert_foreign_key(gold("DimContent"), "section_id", gold("DimSection"), "section_id")
assert_foreign_key(gold("FactAttendanceDay"), "student_id", gold("DimStudent"), "student_id")
assert_foreign_key(gold("FactAttendanceDay"), "date", gold("DimDate"), "date")
assert_foreign_key(gold("FactAssessment"), "assessment_id", gold("DimAssessment"), "assessment_id")
assert_foreign_key(gold("FactAssessment"), "window_id", gold("DimWindow"), "window_id")
assert_foreign_key(gold("FactSubmission"), "content_id", gold("DimContent"), "content_id")
assert_foreign_key(gold("FactStudentAction"), "student_id", gold("DimStudent"), "student_id")
assert_foreign_key(gold("FactStudentAction"), "school_id", gold("DimSchool"), "school_id")
assert_foreign_key(gold("FactStudentAction"), "teacher_id", gold("DimTeacher"), "teacher_id")
assert_foreign_key(gold("FactStudentAction"), "date", gold("DimDate"), "date")
assert_foreign_key(gold("FactEnrollment"), "student_id", gold("DimStudent"), "student_id")
assert_foreign_key(gold("FactEnrollment"), "school_id", gold("DimSchool"), "school_id")


In [ ]:
# Attendance components must reconcile at every scope.
assert_no_rows(
    "FactAttendanceDay day components add up",
    f"""
    SELECT *
    FROM {gold('FactAttendanceDay')}
    WHERE expected_days <> known_days + unknown_days
       OR known_days <> present_days + absent_days
    """,
)
assert_no_rows(
    "AggSchoolAttendance matches the daily fact",
    f"""
    SELECT *
    FROM (SELECT SUM(present_days) AS p, SUM(known_days) AS k, SUM(expected_days) AS e FROM {gold('FactAttendanceDay')}) AS f
    CROSS JOIN (SELECT SUM(present_days) AS p, SUM(known_days) AS k, SUM(expected_days) AS e FROM {gold('AggSchoolAttendance')}) AS a
    WHERE f.p <> a.p OR f.k <> a.k OR f.e <> a.e
    """,
)

# Records must not be dropped or multiplied by their lookups.
assert_row_count(gold("FactAssessment"), spark.table(silver("AssessmentResult")).count())
assert_row_count(gold("FactSubmissionState"), spark.table(silver("SubmissionState")).count())
assert_row_count(gold("FactSubmission"), spark.table(silver("Submission")).count())
assert_row_count(gold("DimContent"), spark.table(silver("LearningContent")).count())
assert_row_count(
    gold("FactStudentAction"),
    spark.sql(f"""
        SELECT * FROM {silver('InterventionAction')}
        WHERE action_date >= DATE '{period_start}' AND action_date < DATE '{period_end}'
    """).count(),
)

# Early-warning and follow-up components must mean exactly what the dictionary says.
assert_no_rows(
    "FactAttendanceDay rolling window stays inside its declared length",
    f"""
    SELECT * FROM {gold('FactAttendanceDay')}
    WHERE rolling_absent_days > rolling_expected_days
       OR rolling_expected_days > {ATTENDANCE_RULE_WINDOW_DAYS}
       OR NOT (rolling_window_complete <=> (rolling_expected_days = {ATTENDANCE_RULE_WINDOW_DAYS}))
    """,
)
assert_no_rows(
    "FactStudentAction window components add up",
    f"""
    SELECT * FROM {gold('FactStudentAction')}
    WHERE before_known_days > before_expected_days OR before_absent_days > before_known_days
       OR after_known_days > after_expected_days OR after_absent_days > after_known_days
       OR before_expected_days > window_days OR after_expected_days > window_days
       OR NOT (is_comparable <=> (before_expected_days = window_days AND after_expected_days = window_days))
       OR NOT (is_completed <=> (status = 'completed'))
    """,
)
assert_no_rows(
    "FactStudentAction coursework windows add up",
    f"""
    SELECT * FROM {gold('FactStudentAction')}
    WHERE before_work_missing > before_work_due OR after_work_missing > after_work_due
       OR before_work_due > work_window_pieces OR after_work_due > work_window_pieces
       OR NOT (is_work_comparable <=> (before_work_due = work_window_pieces AND after_work_due = work_window_pieces))
    """,
)
assert_no_rows(
    "FactStudentAction has one first action per learner, reason category and school year",
    f"""
    SELECT f.student_id, f.reason_category, t.term_id
    FROM {gold('FactStudentAction')} AS f
    LEFT JOIN {silver('AcademicTerm')} AS t
      ON f.date >= t.valid_from AND f.date < t.valid_to
    GROUP BY f.student_id, f.reason_category, t.term_id
    HAVING SUM(CASE WHEN f.is_first_in_category THEN 1 ELSE 0 END) <> 1
    """,
)
# Every membership that touches the period appears once per school year it overlaps, and
# the entry and exit categories are the documented ones.
assert_row_count(
    gold("FactEnrollment"),
    spark.sql(f"""
        SELECT 1
        FROM {silver('SchoolEnrollment')} AS m
        JOIN {silver('AcademicTerm')} AS t
          ON m.valid_from < t.valid_to AND m.valid_to > t.valid_from
        WHERE t.valid_to > DATE '{period_start}' AND t.valid_from < DATE '{period_end}'
    """).count(),
)
assert_no_rows(
    "FactEnrollment uses the documented entry and exit categories",
    f"""
    SELECT * FROM {gold('FactEnrollment')}
    WHERE entry_type NOT IN ('First year in data', 'Continuing', 'Transferred in', 'New')
       OR exit_type NOT IN ('Latest year in data', 'Continuing', 'Transferred out', 'Completed final grade', 'Left')
    """,
)

# Coursework achievement. Each column has to mean exactly what the dictionary says, or a
# performance figure built on it is wrong in a way no row count would reveal.
assert_no_rows(
    "FactSubmission is_graded is exactly a score with a usable mark scheme",
    f"""
    SELECT *
    FROM {gold('FactSubmission')}
    WHERE NOT (is_graded <=> (score IS NOT NULL AND COALESCE(points_possible, 0) > 0))
    """,
)
assert_no_rows(
    "FactSubmission points_possible is a usable denominator where present",
    f"SELECT * FROM {gold('FactSubmission')} WHERE points_possible IS NOT NULL AND points_possible <= 0",
)
assert_no_rows(
    "FactSubmission graded scores sit inside the mark scheme",
    f"""
    SELECT *
    FROM {gold('FactSubmission')}
    WHERE is_graded AND (score < 0 OR score > points_possible)
    """,
)
assert_no_rows(
    "FactSubmission has exactly one latest attempt per obligation",
    f"""
    SELECT obligation_id
    FROM {gold('FactSubmission')}
    GROUP BY obligation_id
    HAVING SUM(CASE WHEN is_latest_attempt THEN 1 ELSE 0 END) <> 1
    """,
)
assert_no_rows(
    "FactSubmission subject is the subject of its own section",
    f"""
    SELECT f.submission_id
    FROM {gold('FactSubmission')} AS f
    JOIN {gold('DimSection')} AS d ON d.section_id = f.section_id
    WHERE NOT (f.subject <=> d.subject)
    """,
)
# Null-safe equality, so a flag that is absent in Silver has to be absent here too rather
# than quietly becoming false.
assert_no_rows(
    "FactSubmission is_late is the Silver flag unchanged, nulls included",
    f"""
    SELECT f.submission_id
    FROM {gold('FactSubmission')} AS f
    JOIN {silver('Submission')} AS s ON s.submission_id = f.submission_id
    WHERE NOT (f.is_late <=> s.source_late)
    """,
)
assert_no_rows(
    "FactAssessment is_scored is exactly the presence of a score",
    f"SELECT * FROM {gold('FactAssessment')} WHERE NOT (is_scored <=> (score IS NOT NULL))",
)

display(spark.sql(f"""
    SELECT
        SUM(expected_days) AS expected_days,
        SUM(known_days) AS known_days,
        SUM(present_days) AS present_days,
        SUM(absent_days) AS absent_days,
        SUM(unknown_days) AS unknown_days,
        ROUND(SUM(present_days) / NULLIF(SUM(known_days), 0), 4) AS attendance_rate,
        ROUND(SUM(known_days) / NULLIF(SUM(expected_days), 0), 4) AS attendance_coverage
    FROM {gold('FactAttendanceDay')}
"""))

# Achievement operands and the ratios they support, so the numbers can be sanity-checked
# and not just the schema. Both ratios are sums divided by sums, over latest attempts
# only, which is what the semantic model computes for a selection.
display(spark.sql(f"""
    SELECT
        COUNT(*) AS submissions,
        SUM(CASE WHEN is_latest_attempt THEN 1 ELSE 0 END) AS latest_attempts,
        SUM(CASE WHEN is_latest_attempt AND is_graded THEN 1 ELSE 0 END) AS graded,
        SUM(CASE WHEN is_latest_attempt AND NOT is_graded THEN 1 ELSE 0 END) AS ungraded,
        ROUND(SUM(CASE WHEN is_latest_attempt AND is_graded THEN 1 ELSE 0 END)
              / NULLIF(SUM(CASE WHEN is_latest_attempt THEN 1 ELSE 0 END), 0), 4) AS graded_work_coverage,
        SUM(CASE WHEN is_latest_attempt AND is_graded THEN score END) AS points_earned,
        SUM(CASE WHEN is_latest_attempt AND is_graded THEN points_possible END) AS points_possible,
        ROUND(SUM(CASE WHEN is_latest_attempt AND is_graded THEN score END)
              / NULLIF(SUM(CASE WHEN is_latest_attempt AND is_graded THEN points_possible END), 0), 4) AS coursework_score_ratio,
        SUM(CASE WHEN is_late THEN 1 ELSE 0 END) AS flagged_late,
        SUM(CASE WHEN is_late IS NULL THEN 1 ELSE 0 END) AS late_flag_absent
    FROM {gold('FactSubmission')}
"""))

# Enrollment movement between school years.
display(spark.sql(f"""
    SELECT school_year, entry_type, exit_type, COUNT(DISTINCT student_id) AS learners
    FROM {gold('FactEnrollment')}
    GROUP BY school_year, entry_type, exit_type
    ORDER BY school_year, entry_type, exit_type
"""))

# Early warning and follow-up, as the semantic model computes them for the whole selection.
display(spark.sql(f"""
    SELECT
        COUNT(DISTINCT student_id) AS learners_meeting_attendance_rule_5_in_{ATTENDANCE_RULE_WINDOW_DAYS}
    FROM {gold('FactAttendanceDay')}
    WHERE rolling_window_complete AND rolling_absent_days >= 5
"""))
display(spark.sql(f"""
    SELECT
        COUNT(*) AS actions,
        COUNT(DISTINCT student_id) AS learners_with_an_action,
        SUM(CASE WHEN status = 'open' THEN 1 ELSE 0 END) AS open_actions,
        SUM(CASE WHEN reason_category = 'Attendance' AND is_first_in_category AND is_comparable THEN 1 ELSE 0 END) AS learners_compared,
        ROUND(SUM(CASE WHEN reason_category = 'Attendance' AND is_first_in_category AND is_comparable THEN before_absent_days END)
              / NULLIF(SUM(CASE WHEN reason_category = 'Attendance' AND is_first_in_category AND is_comparable THEN before_known_days END), 0), 4) AS absence_rate_before,
        ROUND(SUM(CASE WHEN reason_category = 'Attendance' AND is_first_in_category AND is_comparable THEN after_absent_days END)
              / NULLIF(SUM(CASE WHEN reason_category = 'Attendance' AND is_first_in_category AND is_comparable THEN after_known_days END), 0), 4) AS absence_rate_after
    FROM {gold('FactStudentAction')}
"""))

display(spark.sql(f"""
    SELECT
        COUNT(*) AS administrations,
        SUM(CASE WHEN is_scored THEN 1 ELSE 0 END) AS scored,
        SUM(CASE WHEN is_scored THEN 0 ELSE 1 END) AS unscored
    FROM {gold('FactAssessment')}
"""))

display(spark.sql(f"""
    SELECT
        subject,
        SUM(CASE WHEN is_latest_attempt AND is_graded THEN 1 ELSE 0 END) AS graded,
        ROUND(SUM(CASE WHEN is_latest_attempt AND is_graded THEN score END)
              / NULLIF(SUM(CASE WHEN is_latest_attempt AND is_graded THEN points_possible END), 0), 4) AS coursework_score_ratio
    FROM {gold('FactSubmission')}
    GROUP BY subject
    ORDER BY subject
"""))


# Publish

Publish moves the pointer, then retires what it superseded.

Until this cell runs, the previously published snapshot is still complete and still named
by the newest publication row. Only after the new row is appended is the old snapshot
removed, so a failure anywhere earlier leaves a consistent published dataset behind.


In [ ]:
publish_gold_refresh(snapshot_id, as_of, period_start, period_end, refreshed_tables)
retire_superseded_snapshots(refreshed_tables, snapshot_id)
assert_only_published_snapshot(refreshed_tables, snapshot_id)

display(get_table_counts({t: get_gold_full_table_name(t) for t in refreshed_tables}))
